# Análisis Geoestadístico Descriptivo

Dataset: **US_Accidents_Final_FE.csv**

Objetivo: construir un análisis descriptivo con foco geoestadístico, combinando variables espaciales, temporales, meteorológicas, de infraestructura y severidad del accidente.

> Nota: el análisis es descriptivo. Los gráficos permiten detectar patrones y asociaciones, pero no prueban causalidad.


## 1. Librerías y configuración visual

Esta versión combina:

- `plotly`: visualizaciones tipo dashboard.
- `seaborn` y `matplotlib`: gráficos estadísticos estáticos más cuidados.
- `h3`: agregación espacial en celdas hexagonales.
- `missingno`: diagnóstico visual de calidad de datos.



In [ ]:
import warnings
warnings.filterwarnings('ignore')

from pathlib import Path
import math
import os
import re

import numpy as np
import pandas as pd

import matplotlib.pyplot as plt
import seaborn as sns

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.io as pio
import kaleido

import folium
from folium.plugins import HeatMap, MarkerCluster, FastMarkerCluster
from branca.colormap import linear

from sklearn.neighbors import NearestNeighbors
from scipy import stats

# Librerías para visualización 
import missingno as msno
import geopandas as gpd
from shapely.geometry import Polygon
import h3

pd.set_option('display.max_columns', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.3f}')

# Estética global
pio.templates.default = 'plotly_white'
sns.set_theme(style='whitegrid', context='talk')

COLOR_SEVERITY = {
    1: '#2E86AB',
    2: '#F6C85F',
    3: '#F18F01',
    4: '#C73E1D',
}

TABLEAU_LIKE = ['#4E79A7', '#F28E2B', '#E15759', '#76B7B2', '#59A14F', '#EDC948', '#B07AA1', '#FF9DA7', '#9C755F', '#BAB0AC']

# Guardado automático de visualizaciones
# Todas las figuras estáticas generadas por Plotly/Matplotlib se guardan en:
#   ./Outputs_notebook/
# La ruta se calcula desde el directorio desde el que se ejecuta el notebook.
OUTPUT_DIR = Path.cwd() / "Outputs_notebook"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

_PLOT_SAVE_COUNTER = {"plotly": 0, "matplotlib": 0}

def _safe_filename(text, fallback="figure"):
    """Convierte títulos de gráficos en nombres de archivo seguros."""
    if text is None:
        text = fallback
    text = re.sub(r"<[^>]+>", " ", str(text))
    text = re.sub(r"[^A-Za-z0-9ÁÉÍÓÚÜÑáéíóúüñ_ -]+", " ", text)
    text = re.sub(r"\s+", "_", text).strip("_").lower()
    return text[:90] or fallback

def _save_plotly_figure(fig, name=None):
    """Guarda una figura Plotly como PNG."""
    _PLOT_SAVE_COUNTER["plotly"] += 1
    counter = _PLOT_SAVE_COUNTER["plotly"]

    title = getattr(getattr(fig, "layout", None), "title", None)
    title_text = getattr(title, "text", None)

    base = _safe_filename(name or title_text, fallback=f"plotly_{counter:03d}")
    png_path = OUTPUT_DIR / f"{counter:03d}_{base}.png"

    width = int(fig.layout.width) if fig.layout.width else 1600
    height = int(fig.layout.height) if fig.layout.height else 800
    fig.write_image(str(png_path), width=width, height=height, scale=2)
    print(f"Imagen guardada: {png_path}")
    return png_path

def _save_matplotlib_figure(fig=None, name=None):
    """Guarda la figura Matplotlib activa como PNG."""
    _PLOT_SAVE_COUNTER["matplotlib"] += 1
    counter = _PLOT_SAVE_COUNTER["matplotlib"]
    fig = fig or plt.gcf()
    base = _safe_filename(name, fallback=f"matplotlib_{counter:03d}")
    png_path = OUTPUT_DIR / f"mpl_{counter:03d}_{base}.png"

    fig.savefig(str(png_path), dpi=200, bbox_inches="tight")
    print(f"Imagen guardada: {png_path}")
    return png_path

# Parchear Plotly para que cualquier fig.show() guarde una imagen estática automáticamente.
from plotly.basedatatypes import BaseFigure

if not getattr(BaseFigure.show, "_autosave_patched", False):
    _original_plotly_show = BaseFigure.show

    def _autosave_plotly_show(self, *args, **kwargs):
        _save_plotly_figure(self)
        return _original_plotly_show(self, *args, **kwargs)

    _autosave_plotly_show._autosave_patched = True
    BaseFigure.show = _autosave_plotly_show

# Parchear Matplotlib para que cualquier plt.show() guarde una imagen estática automáticamente.
if not getattr(plt.show, "_autosave_patched", False):
    _original_matplotlib_show = plt.show

    def _autosave_matplotlib_show(*args, **kwargs):
        _save_matplotlib_figure()
        return _original_matplotlib_show(*args, **kwargs)

    _autosave_matplotlib_show._autosave_patched = True
    plt.show = _autosave_matplotlib_show


## 1.1. Funciones de estilo

Estas funciones convierten los gráficos del notebook en piezas más parecidas a Power BI/Tableau: fondo limpio, títulos grandes y formato consistente.


In [ ]:
def style_plotly(fig, title=None, subtitle=None, height=520, legend_top=True):
    full_title = title or ''
    if subtitle:
        full_title = f"{full_title}<br><sup>{subtitle}</sup>"

    fig.update_layout(
        title=dict(text=full_title, x=0.02, xanchor='left', font=dict(size=22)),
        height=height,
        plot_bgcolor='white',
        paper_bgcolor='white',
        font=dict(size=14, family='Arial'),
        margin=dict(l=50, r=35, t=85, b=50),
    )
    fig.update_xaxes(showgrid=False, zeroline=False)
    fig.update_yaxes(gridcolor='rgba(0,0,0,0.08)', zeroline=False)
    if legend_top:
        fig.update_layout(legend=dict(orientation='h', yanchor='bottom', y=1.02, xanchor='right', x=1))
    return fig

def top_n_with_other(series, n=12):
    """Agrupa categorías poco frecuentes como 'Other' para gráficos legibles."""
    counts = series.dropna().astype(str).value_counts()
    top = counts.head(n)
    other = counts.iloc[n:].sum()
    if other > 0:
        top.loc['Other'] = other
    return top.reset_index().rename(columns={'index': 'category', series.name or 0: 'count'})


def h3_cell_from_latlng(lat, lng, resolution=7):
    """Compatibilidad entre versiones nuevas y antiguas de h3-py."""
    if pd.isna(lat) or pd.isna(lng):
        return None
    if hasattr(h3, 'latlng_to_cell'):
        return h3.latlng_to_cell(lat, lng, resolution)
    return h3.geo_to_h3(lat, lng, resolution)


def h3_boundary_polygon(cell):
    if pd.isna(cell):
        return None
    if hasattr(h3, 'cell_to_boundary'):
        boundary = h3.cell_to_boundary(cell)
    else:
        boundary = h3.h3_to_geo_boundary(cell)
    # H3 devuelve lat/lng; shapely espera x/y = lng/lat
    return Polygon([(lng, lat) for lat, lng in boundary])


def infer_available(columns):
    return [c for c in columns if c in df.columns]


## 2. Carga del dataset

In [ ]:
DATA_PATH = Path("../../data/US_Accidents_Final_FE.csv")
if not DATA_PATH.exists():
    DATA_PATH = Path("data/US_Accidents_Final_FE.csv")
raw = pd.read_csv(DATA_PATH)
print(f"Dataset: {raw.shape[0]:,} rows x {raw.shape[1]:,} columns")
raw.head()


## 3. Limpieza ligera y creación de variables auxiliares

Se normalizan nombres de columnas, se convierte la fecha, se crean variables temporales y se prepara una versión limpia para visualización espacial.

In [ ]:
def clean_column_name(col: str) -> str:
    return (
        col.strip()
           .replace('...', '')
           .replace('.', '_')
           .replace(' ', '_')
           .replace('-', '_')
           .lower()
    )

df = raw.copy()
df.columns = [clean_column_name(c) for c in df.columns]

# Conversión temporal
if 'start_time' in df.columns:
    df['start_time'] = pd.to_datetime(df['start_time'], errors='coerce')
    df['year'] = df['start_time'].dt.year
    df['month'] = df['start_time'].dt.month
    df['month_name'] = df['start_time'].dt.month_name()
    df['dayofweek'] = df['start_time'].dt.dayofweek
    df['day_name'] = df['start_time'].dt.day_name()
    df['hour'] = df['start_time'].dt.hour
    df['date'] = df['start_time'].dt.date

# Asegurar tipos categóricos útiles
for col in ['severity', 'junction', 'traffic_signal', 'sunrise_sunset', 'weather_group', 'wind_group', 'has_infrastructure']:
    if col in df.columns:
        df[col] = df[col].astype('category')

# Dataset espacial válido
geo = df.dropna(subset=['start_lat', 'start_lng']).copy()
geo = geo[(geo['start_lat'].between(-90, 90)) & (geo['start_lng'].between(-180, 180))].copy()

print(f'Dimensiones tras preparación: {df.shape[0]:,} filas x {df.shape[1]:,} columnas')
print(f'Registros con coordenadas válidas: {geo.shape[0]:,}')
df.head()

## Primera Diapositiva

In [ ]:
# Asegurar que severity sea numérica para medias y comparaciones
df["severity_num"] = pd.to_numeric(df["severity"], errors="coerce")

n_accidents = len(df)
avg_severity = df["severity_num"].mean()
pct_severe = df["severity_num"].ge(3).mean() * 100
n_counties = df["county"].nunique() if "county" in df.columns else np.nan
peak_hour = int(df["hour"].mode().iloc[0]) if "hour" in df.columns and df["hour"].notna().any() else None

kpi_data = pd.DataFrame({
    "metric": [
        "Accidentes",
        "Severidad media",
        "% severidad ≥ 3",
        "Condados",
        "Hora pico"
    ],
    "value": [
        f"{n_accidents:,.0f}".replace(",", "."),
        f"{avg_severity:.2f}",
        f"{pct_severe:.1f}%",
        f"{n_counties:,.0f}".replace(",", ".") if pd.notna(n_counties) else "N/D",
        f"{peak_hour}:00" if peak_hour is not None else "N/D"
    ]
})

fig = go.Figure()

for i, row in kpi_data.iterrows():
    fig.add_trace(go.Indicator(
        mode="number",
        value=i + 1,
        title={
            "text": f"<b>{row['value']}</b><br><span style='font-size:16px'>{row['metric']}</span>"
        },
        domain={
            "row": 0,
            "column": i
        }
    ))

fig.update_layout(
    grid={"rows": 1, "columns": len(kpi_data), "pattern": "independent"},
    height=230,
    paper_bgcolor="white",
    margin=dict(l=20, r=20, t=30, b=20),
)

fig.show()


In [ ]:
sev_counts = df['severity'].value_counts().sort_index().reset_index()
sev_counts.columns = ['severity', 'n']
sev_counts['pct'] = 100 * sev_counts['n'] / sev_counts['n'].sum()

fig = go.Figure()
fig.add_trace(go.Bar(
    x=sev_counts['severity'].astype(str),
    y=sev_counts['n'],
    text=sev_counts['pct'].round(1).astype(str) + '%',
    textposition='outside',
    marker=dict(color=sev_counts['pct'], colorscale='Viridis')
))
fig.update_layout(
    title='Distribución de accidentes por severidad',
    xaxis_title='Severidad',
    yaxis_title='Número de accidentes',
    showlegend=False
)
fig.show()

In [ ]:
if {'day_name', 'hour'}.issubset(df.columns):
    order_days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    pivot_count = df.pivot_table(index='day_name', columns='hour', values='severity', aggfunc='count').reindex(order_days)

    fig = px.imshow(
        pivot_count,
        aspect='auto',
        color_continuous_scale='Viridis',
        title='Heatmap de frecuencia: día de la semana vs hora'
    )
    fig.update_layout(xaxis_title='Hora', yaxis_title='Día de la semana')
    fig.show()

    pivot_sev = df.pivot_table(index='day_name', columns='hour', values='severity', aggfunc=lambda s: pd.to_numeric(s).mean()).reindex(order_days)
    fig = px.imshow(
        pivot_sev,
        aspect='auto',
        color_continuous_scale='Inferno',
        title='Heatmap de severidad media: día de la semana vs hora'
    )
    fig.update_layout(xaxis_title='Hora', yaxis_title='Día de la semana')
    fig.show()

In [ ]:
# Severidad media y % impacto alto por hora
if {'hour', 'severity_num'}.issubset(df.columns):
    hourly_sev = (
        df.groupby('hour')
        .agg(
            accidents=('severity_num', 'size'),
            avg_severity=('severity_num', 'mean'),
            pct_severe=('severity_num', lambda x: (x >= 3).mean() * 100)
        )
        .reset_index()
    )

    fig = make_subplots(specs=[[{'secondary_y': True}]])
    fig.add_trace(go.Bar(
        x=hourly_sev['hour'],
        y=hourly_sev['accidents'],
        name='Accidentes',
        marker_color='rgba(78,121,167,0.55)',
    ), secondary_y=False)
    fig.add_trace(go.Scatter(
        x=hourly_sev['hour'],
        y=hourly_sev['avg_severity'],
        name='Severidad media',
        mode='lines+markers',
        line=dict(width=3, color='#E15759'),
    ), secondary_y=True)
    fig.update_yaxes(title_text='Número de accidentes', secondary_y=False)
    fig.update_yaxes(title_text='Severidad media', secondary_y=True)
    fig.update_xaxes(title_text='Hora del día')
    fig = style_plotly(fig, 'Volumen y severidad no siempre crecen al mismo ritmo', 'Comparación horaria entre frecuencia de accidentes y severidad media', height=560)
    fig.show()


# Heatmap día-hora con severidad media
if {'day_of_week', 'hour', 'severity'}.issubset(df.columns):
    day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
    pivot = df.pivot_table(index='day_of_week', columns='hour', values='severity', aggfunc='mean').reindex(day_order)
    fig = px.imshow(
        pivot,
        aspect='auto',
        color_continuous_scale='OrRd',
        labels=dict(x='Hora', y='Día de la semana', color='Severidad media')
    )
    fig = style_plotly(fig, 'La severidad presenta patrones temporales diferenciados', 'Heatmap de severidad media por día de la semana y hora', height=560, legend_top=False)
    fig.show()


# Weather vs severidad: solo categorías relevantes
if {'weather_condition', 'severity'}.issubset(df.columns):
    top_weather = df['weather_condition'].dropna().astype(str).value_counts().head(10).index
    weather_sev = (
        df[df['weather_condition'].astype(str).isin(top_weather)]
        .groupby('weather_condition')
        .agg(
            accidents=('severity', 'size'),
            avg_severity=('severity', 'mean'),
            pct_severe=('severity', lambda x: (x >= 3).mean() * 100)
        )
        .reset_index()
        .sort_values('pct_severe', ascending=True)
    )

    fig = px.scatter(
        weather_sev,
        x='accidents',
        y='pct_severe',
        size='avg_severity',
        hover_name='weather_condition',
        text='weather_condition',
        color='pct_severe',
        color_continuous_scale='Reds'
    )
    fig.update_traces(textposition='top center')
    fig.update_layout(xaxis_title='Volumen de accidentes', yaxis_title='% accidentes con impacto alto en el tráfico')
    fig = style_plotly(fig, 'El volumen de accidentes y la gravedad no describen el mismo fenómeno', 'Condiciones meteorológicas principales: frecuencia vs porcentaje de accidentes con impacto alto en el tráfico', height=620, legend_top=False)
    fig.show()



In [ ]:
import plotly.graph_objects as go
import pandas as pd

geo["severity_num"] = pd.to_numeric(geo["severity"], errors="coerce")

weather_risk = (
    geo.groupby("weather_group")
    .agg(
        accidents=("weather_group", "size"),
        severe_pct=("severity_num", lambda x: (x >= 3).mean() * 100),
        severity_mean=("severity_num", "mean")
    )
    .reset_index()
)

# Opcional: filtrar categorías con pocos registros
weather_risk = weather_risk[weather_risk["accidents"] >= 30]

weather_risk = weather_risk.sort_values("severe_pct", ascending=True)

fig = go.Figure()

fig.add_trace(go.Scatter(
    x=weather_risk["severe_pct"],
    y=weather_risk["weather_group"],
    mode="markers+text",
    marker=dict(
        size=22,
        color=weather_risk["severe_pct"],
        colorscale="YlOrRd",
        showscale=True,
        colorbar=dict(title="% impacto alto")
    ),
    text=weather_risk["severe_pct"].round(1).astype(str) + "%",
    textposition="middle right",
    hovertemplate=(
        "<b>%{y}</b><br>" +
        "% impacto alto: %{x:.1f}%<br>" +
        "Accidentes: %{customdata[0]:,}<br>" +
        "Severidad media: %{customdata[1]:.2f}<extra></extra>"
    ),
    customdata=weather_risk[["accidents", "severity_mean"]].values
))

for _, row in weather_risk.iterrows():
    fig.add_shape(
        type="line",
        x0=0,
        x1=row["severe_pct"],
        y0=row["weather_group"],
        y1=row["weather_group"],
        line=dict(
            color="rgba(80,80,80,0.25)",
            width=4
        )
    )

fig.update_layout(
    title=dict(
        text="La nieve concentra el mayor porcentaje de accidentes con impacto alto en el tráfico",
        x=0.02,
        xanchor="left",
        font=dict(size=24)
    ),
    xaxis=dict(
        title="% de accidentes con severity ≥ 3",
        range=[0, weather_risk["severe_pct"].max() * 1.18],
        ticksuffix="%",
        gridcolor="rgba(0,0,0,0.08)"
    ),
    yaxis=dict(
        title="",
        categoryorder="array",
        categoryarray=weather_risk["weather_group"].tolist()
    ),
    height=540,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=120, r=80, t=90, b=60),
    font=dict(size=15),
    showlegend=False
)

fig.show()



# Gráficos adicionales que no se usaron para el dashboard

In [ ]:
num_cols = [
    c for c in ['humidity', 'pressure_in', 'wind_speed_mph', 'precipitation_in',
                'temperature_c', 'wind_chill_c', 'start_lat', 'start_lng']
    if c in df.columns
]

num_summary = df[num_cols].describe().T
num_summary['missing_%'] = df[num_cols].isna().mean() * 100
num_summary['skewness'] = df[num_cols].skew(numeric_only=True)
num_summary['kurtosis'] = df[num_cols].kurtosis(numeric_only=True)
num_summary.round(3)

In [ ]:
for col in num_cols:
    fig = px.histogram(
        df,
        x=col,
        nbins=45,
        marginal='box',
        title=f'Distribución univariante de {col}',
        color_discrete_sequence=px.colors.sequential.Viridis[:1]
    )
    fig.update_layout(xaxis_title=col, yaxis_title='Frecuencia')
    fig.show()

### Variables categóricas: frecuencias y concentración



In [ ]:
def plot_top_categories(data, col, top_n=15):
    tmp = data[col].astype(str).value_counts().head(top_n).sort_values().reset_index()
    tmp.columns = [col, 'n']
    fig = px.bar(
        tmp,
        x='n',
        y=col,
        orientation='h',
        title=f'Top categorías de {col}',
        text='n',
        color='n',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(yaxis_title='', xaxis_title='Frecuencia', coloraxis_showscale=False)
    fig.show()

for col in ['county', 'weather_group', 'wind_group', 'sunrise_sunset']:
    if col in df.columns:
        plot_top_categories(df, col, top_n=15)

In [ ]:
if 'hour' in df.columns:
    hour_counts = df.groupby('hour').size().reset_index(name='accidents')
    fig = px.bar(
        hour_counts,
        x='hour',
        y='accidents',
        title='Accidentes por hora del día',
        color='accidents',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(xaxis_title='Hora', yaxis_title='Accidentes', coloraxis_showscale=False)
    fig.show()

if 'day_name' in df.columns:
    order_days = ['Monday','Tuesday','Wednesday','Thursday','Friday','Saturday','Sunday']
    day_counts = df['day_name'].value_counts().reindex(order_days).reset_index()
    day_counts.columns = ['day_name', 'accidents']
    fig = px.bar(
        day_counts,
        x='day_name',
        y='accidents',
        title='Accidentes por día de la semana',
        color='accidents',
        color_continuous_scale='Viridis'
    )
    fig.update_layout(xaxis_title='Día', yaxis_title='Accidentes', coloraxis_showscale=False)
    fig.show()

In [ ]:
severity_num_cols = [c for c in ['temperature_c', 'wind_chill_c', 'humidity', 'pressure_in', 'wind_speed_mph', 'precipitation_in'] if c in df.columns]

for col in severity_num_cols:
    fig = px.violin(
        df,
        x='severity',
        y=col,
        box=True,
        points='outliers',
        color='severity',
        title=f'{col} según severidad del accidente'
    )
    fig.update_layout(xaxis_title='Severidad', yaxis_title=col, showlegend=False)
    fig.show()

In [ ]:
df["severity_num"] = pd.to_numeric(df["severity"], errors="coerce")

infra_vars = ["junction", "traffic_signal", "has_infrastructure"]

infra_long = []

for var in infra_vars:
    if var in df.columns:
        temp = (
            df.groupby(var)
            .agg(
                accidents=("severity_num", "size"),
                severity_mean=("severity_num", "mean"),
                severe_pct=("severity_num", lambda x: (x >= 3).mean() * 100)
            )
            .reset_index()
        )
        temp["variable"] = var
        temp["status"] = temp[var].astype(str)
        infra_long.append(
            temp[["variable", "status", "accidents", "severity_mean", "severe_pct"]]
        )

infra_summary = pd.concat(infra_long, ignore_index=True)

label_var = {
    "junction": "Junction",
    "traffic_signal": "Traffic signal",
    "has_infrastructure": "Infrastructure"
}

label_status = {
    "True": "Sí",
    "False": "No",
    "1": "Sí",
    "0": "No"
}

infra_summary["variable_label"] = infra_summary["variable"].map(label_var)
infra_summary["status_label"] = infra_summary["status"].map(label_status).fillna(infra_summary["status"])

order = ["Junction", "Traffic signal", "Infrastructure"]
infra_summary["variable_label"] = pd.Categorical(
    infra_summary["variable_label"],
    categories=order,
    ordered=True
)

infra_summary = infra_summary.sort_values("variable_label")

fig = go.Figure()

# Líneas conectoras entre No y Sí
for var in order:
    temp = infra_summary[infra_summary["variable_label"] == var]
    if {"No", "Sí"}.issubset(set(temp["status_label"])):
        no_val = temp.loc[temp["status_label"] == "No", "severe_pct"].iloc[0]
        yes_val = temp.loc[temp["status_label"] == "Sí", "severe_pct"].iloc[0]
        
        fig.add_trace(go.Scatter(
            x=[yes_val, no_val],
            y=[var, var],
            mode="lines",
            line=dict(color="rgba(80,80,80,0.35)", width=4),
            showlegend=False,
            hoverinfo="skip"
        ))

# Puntos No
no_df = infra_summary[infra_summary["status_label"] == "No"]
fig.add_trace(go.Scatter(
    x=no_df["severe_pct"],
    y=no_df["variable_label"],
    mode="markers+text",
    name="No",
    marker=dict(
        size=18,
        color="#A8DADC",
        line=dict(color="white", width=1.5)
    ),
    text=no_df["severe_pct"].round(1).astype(str) + "%",
    textposition="middle right",
    customdata=no_df[["accidents", "severity_mean"]].values,
    hovertemplate=(
        "<b>%{y} — No</b><br>" +
        "% impacto alto: %{x:.1f}%<br>" +
        "Accidentes: %{customdata[0]:,}<br>" +
        "Severidad media: %{customdata[1]:.2f}<extra></extra>"
    )
))

# Puntos Sí
yes_df = infra_summary[infra_summary["status_label"] == "Sí"]
fig.add_trace(go.Scatter(
    x=yes_df["severe_pct"],
    y=yes_df["variable_label"],
    mode="markers+text",
    name="Sí",
    marker=dict(
        size=18,
        color="#E76F51",
        line=dict(color="white", width=1.5)
    ),
    text=yes_df["severe_pct"].round(1).astype(str) + "%",
    textposition="middle left",
    customdata=yes_df[["accidents", "severity_mean"]].values,
    hovertemplate=(
        "<b>%{y} — Sí</b><br>" +
        "% impacto alto: %{x:.1f}%<br>" +
        "Accidentes: %{customdata[0]:,}<br>" +
        "Severidad media: %{customdata[1]:.2f}<extra></extra>"
    )
))

fig.update_layout(
    title=dict(
        text="Los accidentes con impacto alto en el tráfico son proporcionalmente más frecuentes fuera de infraestructuras reguladas",
        x=0.02,
        xanchor="left",
        font=dict(size=23, color="#243B5A")
    ),
    height=460,
    width=1050,
    plot_bgcolor="white",
    paper_bgcolor="white",
    margin=dict(l=140, r=80, t=85, b=60),
    xaxis=dict(
        title="% de accidentes con impacto alto en el tráfico",
        ticksuffix="%",
        gridcolor="rgba(0,0,0,0.08)",
        range=[
            max(0, infra_summary["severe_pct"].min() - 2),
            infra_summary["severe_pct"].max() + 2
        ]
    ),
    yaxis=dict(
        title="",
        autorange="reversed"
    ),
    legend=dict(
        title="Presencia",
        orientation="h",
        yanchor="bottom",
        y=1.03,
        xanchor="right",
        x=1
    ),
    font=dict(size=15, color="#243B5A")
)

fig.show()



In [ ]:
corr_cols = [c for c in ['severity', 'humidity', 'pressure_in', 'wind_speed_mph', 'precipitation_in',
                         'temperature_c', 'wind_chill_c', 'start_lat', 'start_lng', 'hour'] if c in df.columns]

corr_df = df[corr_cols].copy()
for c in corr_df.columns:
    corr_df[c] = pd.to_numeric(corr_df[c], errors='coerce')

corr = corr_df.corr(method='spearman')

fig = px.imshow(
    corr,
    text_auto='.2f',
    color_continuous_scale='RdBu_r',
    zmin=-1,
    zmax=1,
    title='Matriz de correlación Spearman'
)
fig.update_layout(width=900, height=750)
fig.show()

# Segunda Diapositiva: Geoestadística 

In [ ]:
fig = px.density_mapbox(
    geo,
    lat='start_lat',
    lon='start_lng',
    z=None,
    radius=6,
    zoom=7,
    height=750,
    width=1200,
    title='Mapa de densidad espacial de accidentes'
)
fig.update_layout(mapbox_style='open-street-map', margin={'r':0,'t':50,'l':0,'b':0})
fig.show()

### Agregación por condado



In [ ]:
if 'county' in df.columns:
    county_stats = (
        df.groupby('county')
          .agg(
              accidents=('severity', 'size'),
              severity_mean=('severity', lambda s: pd.to_numeric(s, errors='coerce').mean()),
              temperature_mean=('temperature_c', 'mean') if 'temperature_c' in df.columns else ('severity', 'size'),
              wind_mean=('wind_speed_mph', 'mean') if 'wind_speed_mph' in df.columns else ('severity', 'size'),
              precipitation_mean=('precipitation_in', 'mean') if 'precipitation_in' in df.columns else ('severity', 'size'),
              lat_mean=('start_lat', 'mean'),
              lng_mean=('start_lng', 'mean')
          )
          .reset_index()
          .sort_values('accidents', ascending=False)
    )
    display(county_stats.head(20))

In [ ]:
if 'county_stats' in globals():
    fig = px.scatter_mapbox(
        county_stats,
        lat='lat_mean',
        lon='lng_mean',
        size='accidents',
        color='severity_mean',
        hover_name='county',
        hover_data=['accidents', 'severity_mean'],
        color_continuous_scale='Inferno',
        zoom=7,
        height=700,
        title='Centroides por condado: tamaño = accidentes, color = severidad media'
    )
    fig.update_layout(mapbox_style='open-street-map', margin={'r':0,'t':50,'l':0,'b':0})
    fig.show()

### Mapas bivariantes H3: exposición geográfica y severidad

Estos mapas cruzan ubicación geográfica con una segunda variable de contexto y una métrica de gravedad. El color representa la exposición principal de cada zona y el tamaño representa el porcentaje de accidentes con impacto alto en el tráfico, por lo que ya no se interpreta solo dónde hay más accidentes, sino dónde coinciden condiciones de riesgo y mayor severidad.


In [ ]:
geo["severity_num"] = pd.to_numeric(geo["severity"], errors="coerce")

required_cols = {"start_lat", "start_lng", "severity_num", "sunrise_sunset"}


h3_night_data = geo.dropna(
    subset=["start_lat", "start_lng", "severity_num", "sunrise_sunset"]
).copy()

def latlng_to_h3_cell(lat, lng, resolution=7):
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lng, resolution)
    return h3.geo_to_h3(lat, lng, resolution)

if "h3_cell" not in h3_night_data.columns:
    h3_night_data["h3_cell"] = h3_night_data.apply(
        lambda row: latlng_to_h3_cell(row["start_lat"], row["start_lng"], 7),
        axis=1
    )

    h3_night = (
        h3_night_data
        .groupby("h3_cell")
        .agg(
            accidents=("severity_num", "size"),
            night_pct=("sunrise_sunset", lambda x: (x.astype(str) == "Night").mean() * 100),
            severe_pct=("severity_num", lambda x: (x >= 3).mean() * 100),
            severity_mean=("severity_num", "mean"),
            lat=("start_lat", "mean"),
            lng=("start_lng", "mean")
        )
        .reset_index()
    )

h3_night = h3_night[h3_night["accidents"] >= 5].copy()
h3_night["severity_marker"] = h3_night["severe_pct"].clip(lower=2)

# Color = % de accidentes nocturnos; tamano = % de accidentes con impacto alto en el tráfico.
h3_night = h3_night.sort_values(
    ["night_pct", "severity_marker"],
    ascending=[True, True]
).copy()

night_colorscale = [
    [0.00, "#f7fbff"],
    [0.20, "#c6dbef"],
    [0.40, "#6baed6"],
    [0.60, "#fdae61"],
    [0.80, "#e34a33"],
    [1.00, "#4d0012"],
]

color_max = max(80, h3_night["night_pct"].quantile(0.95))
center_lat = h3_night["lat"].mean() - 0.20
center_lng = h3_night["lng"].mean()

fig = px.scatter_mapbox(
    h3_night,
    lat="lat",
    lon="lng",
    size="severity_marker",
    color="night_pct",
    color_continuous_scale=night_colorscale,
    range_color=(0, color_max),
    size_max=30,
    hover_data={
        "accidents": True,
        "night_pct": ":.1f",
        "severe_pct": ":.1f",
        "severity_mean": ":.2f",
        "severity_marker": False,
        "lat": False,
        "lng": False,
        "h3_cell": False
    },
    center=dict(lat=center_lat, lon=center_lng),
    zoom=7.3,
    height=750,
    width=1200,
    title="Nocturnidad x severidad por zona H3"
)

fig.update_traces(
    marker=dict(
        sizemin=5,
        opacity=0.88
    )
)

fig.update_layout(
    mapbox_style="carto-positron",
    margin=dict(l=0, r=0, t=60, b=0),
    coloraxis_colorbar=dict(title="% noche", ticksuffix="%"),
    legend_title_text="% impacto alto"
)

fig.show()

In [ ]:
import pandas as pd
import plotly.express as px
import h3

# Mapa bivariante: clima adverso x severidad por zona H3.
# Color = % de registros con clima adverso
# Tamaño = % de accidentes con impacto alto en el tráfico

geo["severity_num"] = pd.to_numeric(geo["severity"], errors="coerce")

required_cols = {"start_lat", "start_lng", "severity_num", "weather_group"}
missing_cols = required_cols - set(geo.columns)

h3_weather = geo.dropna(
    subset=["start_lat", "start_lng", "severity_num", "weather_group"]
).copy()

def latlng_to_h3_cell(lat, lng, resolution=7):
    if hasattr(h3, "latlng_to_cell"):
        return h3.latlng_to_cell(lat, lng, resolution)
    return h3.geo_to_h3(lat, lng, resolution)

# Crear columna H3 si no existe
if "h3_cell" not in h3_weather.columns:
    h3_weather["h3_cell"] = h3_weather.apply(
        lambda row: latlng_to_h3_cell(
            row["start_lat"],
            row["start_lng"],
            resolution=7
        ),
        axis=1
    )

    # Define clima adverso.
    # Puedes ajustar esta lista si quieres considerar Cloudy como clima adverso.
    clear_weather = {"Clear", "Cloudy"}
    h3_weather["is_bad_weather"] = ~h3_weather["weather_group"].isin(clear_weather)

    def most_common_value(values):
        values = values.dropna().astype(str)
        if values.empty:
            return "Sin dato"
        return values.mode().iloc[0]

    h3_weather_map = (
        h3_weather
        .groupby("h3_cell")
        .agg(
            accidents=("severity_num", "size"),
            bad_weather_pct=("is_bad_weather", lambda x: x.mean() * 100),
            severe_pct=("severity_num", lambda x: (x >= 3).mean() * 100),
            severity_mean=("severity_num", "mean"),
            dominant_weather=("weather_group", most_common_value),
            lat=("start_lat", "mean"),
            lng=("start_lng", "mean")
        )
        .reset_index()
    )

    # Filtrar celdas con pocos accidentes para evitar porcentajes muy inestables
    h3_weather_map = h3_weather_map[h3_weather_map["accidents"] >= 5].copy()


    # Tamaño = % de accidentes con impacto alto en el tráfico
    h3_weather_map["severity_marker"] = h3_weather_map["severe_pct"].clip(lower=2)

    # Orden de pintado:
    # Plotly dibuja primero las filas iniciales y deja encima las últimas.
    # Así, un punto pequeño con color más intenso queda por encima de uno grande más claro.
    h3_weather_map = h3_weather_map.sort_values(
        ["bad_weather_pct", "severity_marker"],
        ascending=[True, True]
    ).copy()

    # Escala distinta a nocturnidad: verde -> amarillo -> naranja/rojo
    weather_colorscale = [
        [0.00, "#F7FCF5"],
        [0.20, "#C7E9C0"],
        [0.40, "#74C476"],
        [0.60, "#FEE08B"],
        [0.80, "#F46D43"],
        [1.00, "#7F0000"],
    ]

    # Usar percentil 90 da más contraste visual que llevar la escala hasta valores extremos
    color_max = max(40, h3_weather_map["bad_weather_pct"].quantile(0.90))

    center_lat = h3_weather_map["lat"].mean() - 0.20
    center_lng = h3_weather_map["lng"].mean()

    fig = px.scatter_mapbox(
        h3_weather_map,
        lat="lat",
        lon="lng",
        size="severity_marker",
        color="bad_weather_pct",
        color_continuous_scale=weather_colorscale,
        range_color=(0, color_max),
        size_max=30,
        hover_data={
            "accidents": True,
            "bad_weather_pct": ":.1f",
            "severe_pct": ":.1f",
            "severity_mean": ":.2f",
            "dominant_weather": True,
            "severity_marker": False,
            "lat": False,
            "lng": False,
            "h3_cell": False
        },
        center=dict(lat=center_lat, lon=center_lng),
        zoom=7.4,
        height=700,
        width=1200,
        title="Clima adverso x severidad por zona H3"
    )

    fig.update_traces(
        marker=dict(
            sizemin=5,
            opacity=0.88
        )
    )

    fig.update_layout(
        mapbox_style="carto-positron",
        margin=dict(l=0, r=0, t=60, b=0),
        coloraxis_colorbar=dict(
            title="% clima adverso",
            ticksuffix="%"
        ),
        legend_title_text="% impacto alto"
    )

    fig.show()

# Tercera Diapositiva



In [ ]:
if {'start_lat', 'start_lng', 'severity_num'}.issubset(df.columns):
    H3_RESOLUTION = 6 
    geo_df = df[['start_lat', 'start_lng', 'severity_num']].dropna().copy()
    geo_df = geo_df[geo_df['start_lat'].between(-90, 90) & geo_df['start_lng'].between(-180, 180)]

    if len(geo_df) > 300_000:
        geo_df = geo_df.sample(300_000, random_state=42)

    geo_df['h3_cell'] = geo_df.apply(lambda r: h3_cell_from_latlng(r['start_lat'], r['start_lng'], H3_RESOLUTION), axis=1)
    geo_df = geo_df.dropna(subset=['h3_cell'])

    h3_summary = (
        geo_df.groupby('h3_cell')
        .agg(
            accidents=('severity_num', 'size'),
            avg_severity=('severity_num', 'mean'),
            pct_severe=('severity_num', lambda x: (x >= 3).mean() * 100),
            lat=('start_lat', 'mean'),
            lng=('start_lng', 'mean')
        )
        .reset_index()
    )

    # Versión Plotly tipo dashboard
    fig = px.scatter_mapbox(
        h3_summary,
        lat='lat',
        lon='lng',
        size='accidents',
        color='pct_severe',
        color_continuous_scale='Reds',
        hover_data={'h3_cell': True, 'accidents': ':,', 'avg_severity': ':.2f', 'pct_severe': ':.1f', 'lat': False, 'lng': False},
        zoom=8,
        height=720,
    )
    fig.update_layout(
        mapbox_style='carto-positron',
        title=dict(text=f'Zonas críticas agregadas por H3 ', x=0.02, font=dict(size=24)),
        margin=dict(l=0, r=0, t=70, b=0),
        coloraxis_colorbar=dict(title='% impacto alto')
    )
    fig.show()
    display(h3_summary.sort_values('accidents', ascending=False).head(10))


In [ ]:
# Ranking Top 5 zonas criticas H3 con nombres identificables
# Criterio: zonas con al menos 10 accidentes, ordenadas por % de impacto alto y volumen.
import pandas as pd
import plotly.graph_objects as go

if 'h3_summary' in globals() and not h3_summary.empty:
    h3_rank = h3_summary.copy()

    rename_map = {'severe_pct': 'pct_severe', 'severity_mean': 'avg_severity'}
    h3_rank = h3_rank.rename(columns={k: v for k, v in rename_map.items() if k in h3_rank.columns and v not in h3_rank.columns})

    required_cols = {'h3_cell', 'accidents', 'pct_severe', 'avg_severity', 'lat', 'lng'}
    missing_cols = required_cols - set(h3_rank.columns)

    if missing_cols:
        print('Faltan columnas para construir el ranking:', missing_cols)
    else:
        label_cols = [c for c in ['city', 'county', 'state', 'street'] if c in df.columns]

        if label_cols and {'start_lat', 'start_lng'}.issubset(df.columns):
            label_df = df[['start_lat', 'start_lng'] + label_cols].dropna(subset=['start_lat', 'start_lng']).copy()
            label_df['h3_cell'] = label_df.apply(
                lambda r: h3_cell_from_latlng(r['start_lat'], r['start_lng'], globals().get('H3_RESOLUTION', 6)),
                axis=1
            )

            def most_common_value(s):
                s = s.dropna().astype(str)
                return s.mode().iloc[0] if not s.empty else None

            zone_names = (
                label_df
                .groupby('h3_cell')
                .agg({col: most_common_value for col in label_cols})
                .reset_index()
            )

            def build_zone_label(row):
                parts = []
                if 'city' in row and pd.notna(row['city']):
                    parts.append(str(row['city']))
                if 'county' in row and pd.notna(row['county']):
                    parts.append(str(row['county']) + ' County')
                if 'state' in row and pd.notna(row['state']):
                    parts.append(str(row['state']))
                return ' / '.join(parts[:3]) if parts else row['h3_cell']

            zone_names['zone_name'] = zone_names.apply(build_zone_label, axis=1)
            h3_rank = h3_rank.merge(zone_names[['h3_cell', 'zone_name']], on='h3_cell', how='left')
        else:
            h3_rank['zone_name'] = None

        h3_rank['zone_name'] = h3_rank['zone_name'].fillna(
            h3_rank['lat'].round(3).astype(str) + ', ' + h3_rank['lng'].round(3).astype(str)
        )

        min_accidents = 10
        h3_top5 = (
            h3_rank[h3_rank['accidents'] >= min_accidents]
            .sort_values(['pct_severe', 'accidents'], ascending=[False, False])
            .head(5)
            .copy()
        )

        if h3_top5.empty:
            print(f'No hay zonas H3 con al menos {min_accidents} accidentes.')
        else:
            h3_top5['zona'] = [f'Z{i} - {name}' for i, name in enumerate(h3_top5['zone_name'], start=1)]
            h3_top5['map_label'] = [f'Z{i}: {name}' for i, name in enumerate(h3_top5['zone_name'], start=1)]
            h3_top5['pct_label'] = h3_top5['pct_severe'].round(1).astype(str) + '%'

            fig = go.Figure()
            fig.add_trace(go.Bar(
                x=h3_top5['pct_severe'],
                y=h3_top5['zona'],
                orientation='h',
                marker=dict(
                    color=h3_top5['pct_severe'],
                    colorscale='Reds',
                    cmin=0,
                    cmax=max(100, h3_top5['pct_severe'].max()),
                    line=dict(color='rgba(70,70,70,0.25)', width=1),
                ),
                text=h3_top5['pct_label'],
                textposition='outside',
                customdata=h3_top5[['h3_cell', 'accidents', 'avg_severity', 'lat', 'lng']].values,
                hovertemplate=(
                    '<b>%{y}</b><br>' +
                    'Celda H3: %{customdata[0]}<br>' +
                    '% impacto alto: %{x:.1f}%<br>' +
                    'Accidentes: %{customdata[1]:,}<br>' +
                    'Severidad media: %{customdata[2]:.2f}<br>' +
                    'Lat/Lng: %{customdata[3]:.3f}, %{customdata[4]:.3f}<extra></extra>'
                )
            ))

            fig.update_layout(
                title=dict(
                    text='Top 5 zonas criticas H3 por porcentaje de accidentes con impacto alto en el tráfico',
                    x=0.02,
                    xanchor='left',
                    font=dict(size=24)
                ),
                xaxis_title='% de accidentes con severity >= 3',
                yaxis_title='',
                yaxis=dict(autorange='reversed'),
                height=560,
                width=1250,
                plot_bgcolor='white',
                paper_bgcolor='white',
                margin=dict(l=280, r=120, t=90, b=70),
                showlegend=False
            )

            fig.show()


            fig_map = go.Figure()
            fig_map.add_trace(go.Scattermapbox(
                lat=h3_rank['lat'],
                lon=h3_rank['lng'],
                mode='markers',
                marker=dict(
                    size=(h3_rank['accidents'] ** 0.5) * 2,
                    color=h3_rank['pct_severe'],
                    colorscale='Reds',
                    cmin=0,
                    cmax=max(60, h3_rank['pct_severe'].max()),
                    opacity=0.35,
                    colorbar=dict(title='% impacto alto')
                ),
                text=h3_rank['zone_name'],
                hovertemplate=(
                    '<b>%{text}</b><br>' +
                    '% impacto alto: %{marker.color:.1f}%<br>' +
                    'Accidentes: %{customdata[0]:,}<extra></extra>'
                ),
                customdata=h3_rank[['accidents']].values,
                name='Zonas H3'
            ))

            fig_map.add_trace(go.Scattermapbox(
                lat=h3_top5['lat'],
                lon=h3_top5['lng'],
                mode='markers+text',
                marker=dict(size=18, color='black', opacity=0.9),
                text=h3_top5['map_label'],
                textposition='top right',
                hovertemplate=(
                    '<b>%{text}</b><br>' +
                    '% impacto alto: %{customdata[0]:.1f}%<br>' +
                    'Accidentes: %{customdata[1]:,}<br>' +
                    'Severidad media: %{customdata[2]:.2f}<extra></extra>'
                ),
                customdata=h3_top5[['pct_severe', 'accidents', 'avg_severity']].values,
                name='Top 5'
            ))

            fig_map.update_layout(
                title=dict(
                    text='Top 5 zonas criticas H3 identificadas en el mapa',
                    x=0.02,
                    xanchor='left',
                    font=dict(size=24)
                ),
                mapbox=dict(
                    style='carto-positron',
                    center=dict(lat=h3_top5['lat'].mean(), lon=h3_top5['lng'].mean()),
                    zoom=7
                ),
                height=650,
                width=1250,
                margin=dict(l=0, r=0, t=70, b=0)
            )

            fig_map.show()


            display(
                h3_top5[['zona', 'h3_cell', 'accidents', 'pct_severe', 'avg_severity', 'lat', 'lng']]
                .rename(columns={
                    'zona': 'Zona identificada',
                    'h3_cell': 'Celda H3',
                    'accidents': 'Accidentes',
                    'pct_severe': '% impacto alto',
                    'avg_severity': 'Severidad media',
                    'lat': 'Latitud',
                    'lng': 'Longitud'
                })
                .round({
                    '% impacto alto': 1,
                    'Severidad media': 2,
                    'Latitud': 3,
                    'Longitud': 3
                })
            )
else:
    print('Ejecuta primero el bloque del Mapa H3 para crear h3_summary.')
